# Syft + Grype for Agentic AI Security
**Run this online (Google Colab or any Linux Jupyter with internet).** It downloads the real tools and the live vulnerability database.

## The story
An AI agent is not just a model. It is a model plus dozens of Python packages: LangChain, requests, PyYAML, Jinja2. The agent can run tools, read files and call APIs. So if one of those packages has a hole, an attacker can use the agent to get in.

In 2023, LangChain had a bug (CVE-2023-29374). A user could type a prompt that made the agent run any Python code on the server. Nobody "hacked" the model. They used an old package.

## Two tools, two jobs
- **Syft** makes a list of every ingredient in your agent. This list is called an **SBOM** (Software Bill of Materials).
- **Grype** checks that list against a public database of known security holes (CVEs) and tells you which ones you have and which version fixes them.

Think of it like food: Syft prints the ingredient label, Grype checks it against the recall list.

**Fun fact:** "Bill of materials" comes from factories, where it lists every part needed to build a car. SBOMs became mandatory for US government software suppliers after the 2020 SolarWinds hack, through Executive Order 14028 (May 2021). Both tools are free and open source, made by Anchore.

## Step 1. Install Syft and Grype (real binaries from GitHub)

In [ ]:
!mkdir -p bin
!curl -sSfL https://raw.githubusercontent.com/anchore/syft/main/install.sh  | sh -s -- -b ./bin
!curl -sSfL https://raw.githubusercontent.com/anchore/grype/main/install.sh | sh -s -- -b ./bin

import os
os.environ["PATH"] = os.path.abspath("bin") + ":" + os.environ["PATH"]
!syft version | head -3
!grype version | head -3

## Step 2. Build a small but real AI agent project
A customer-support agent that uses LangChain, loads YAML config and renders Jinja2 prompt templates.
The `requirements.txt` uses older versions, the kind you find in real repos copied from a 2023 tutorial.

In [ ]:
import os, textwrap
os.makedirs("agent_app", exist_ok=True)

open("agent_app/requirements.txt", "w").write(textwrap.dedent("""
    langchain==0.0.171
    openai==0.27.0
    requests==2.25.1
    PyYAML==5.3.1
    Jinja2==2.11.2
    urllib3==1.26.4
    aiohttp==3.8.1
    pydantic==1.10.2
""").strip())

open("agent_app/agent.py", "w").write(textwrap.dedent("""
    import yaml, requests
    from jinja2 import Template
    from langchain.agents import initialize_agent, Tool
    from langchain.llms import OpenAI

    cfg = yaml.load(open("config.yaml"), Loader=yaml.FullLoader)   # risky loader
    prompt = Template(cfg["system_prompt"])

    def order_status(order_id: str) -> str:
        return requests.get(f"{cfg['api']}/orders/{order_id}").text

    tools = [Tool(name="order_status", func=order_status, description="Look up an order")]
    agent = initialize_agent(tools, OpenAI(temperature=0), agent="zero-shot-react-description")
""").strip())

open("agent_app/config.yaml", "w").write('system_prompt: "You are a support agent for {{ company }}"\napi: https://api.example.com\n')
print(open("agent_app/requirements.txt").read())

## Step 3. Syft: create the SBOM (the ingredient label)
Syft reads the project and writes the list in **CycloneDX**, an industry standard format other tools can read.

In [ ]:
!syft dir:agent_app -o table -o cyclonedx-json=sbom.cdx.json

In [ ]:
import json, pandas as pd
sbom = json.load(open("sbom.cdx.json"))
comps = pd.DataFrame([{"name": c["name"], "version": c.get("version"), "purl": c.get("purl")}
                      for c in sbom.get("components", [])])
print("SBOM format:", sbom["bomFormat"], sbom["specVersion"], "| components:", len(comps))
comps

## Step 4. Grype: check the SBOM against the live vulnerability database
First run downloads the latest CVE database from Anchore. That is why this notebook needs internet.
We scan the SBOM, not the code. In real teams, the SBOM is made once at build time and re-scanned every day, because new CVEs appear even when your code does not change.

In [ ]:
!grype sbom:sbom.cdx.json -o table
!grype sbom:sbom.cdx.json -o json --file grype.json -q

## Step 5. Turn the findings into a security report

In [ ]:
g = json.load(open("grype.json"))
rows = []
for m in g["matches"]:
    v, a = m["vulnerability"], m["artifact"]
    rows.append({"package": a["name"], "installed": a["version"], "cve": v["id"],
                 "severity": v["severity"], "fixed_in": ", ".join(v.get("fix", {}).get("versions", [])),
                 "fix_state": v.get("fix", {}).get("state")})
df = pd.DataFrame(rows)
order = ["Critical", "High", "Medium", "Low", "Negligible", "Unknown"]
df["severity"] = pd.Categorical(df["severity"], order, ordered=True)
df = df.sort_values(["severity", "package"])

print("Total findings:", len(df))
print(df["severity"].value_counts().reindex(order).fillna(0).astype(int).to_string())
df.groupby("package")["cve"].count().sort_values(ascending=False).plot.barh(title="CVEs per package", figsize=(7,3));

In [ ]:
# The ones that matter for an agent: Critical and High
df[df["severity"].isin(["Critical", "High"])].reset_index(drop=True)

## Step 6. Security gate (what runs in CI/CD)
`--fail-on high` makes Grype exit with code 1 if anything High or Critical is found. Your pipeline stops and the agent is not deployed.

In [ ]:
import subprocess
r = subprocess.run(["grype", "sbom:sbom.cdx.json", "--fail-on", "high", "-q", "-o", "table"], capture_output=True, text=True)
print("Exit code:", r.returncode, "->", "BLOCK DEPLOYMENT" if r.returncode else "OK TO DEPLOY")

## Step 7. Fix it: upgrade to the versions Grype recommends, then re-scan
We take the highest "fixed in" version per package from Grype's own output. No guessing.

In [ ]:
from packaging.version import Version, InvalidVersion

def safe_v(x):
    try: return Version(x)
    except InvalidVersion: return None

fix = {}
for _, r in df.iterrows():
    for fv in filter(None, (s.strip() for s in r["fixed_in"].split(","))):
        v = safe_v(fv)
        if v and (r["package"].lower() not in fix or v > fix[r["package"].lower()]):
            fix[r["package"].lower()] = v

new_lines = []
for line in open("agent_app/requirements.txt").read().splitlines():
    name, ver = line.split("==")
    new_ver = fix.get(name.lower())
    new_lines.append(f"{name}=={new_ver}" if new_ver and new_ver > Version(ver) else line)

os.makedirs("agent_app_fixed", exist_ok=True)
open("agent_app_fixed/requirements.txt", "w").write("\n".join(new_lines))
print(pd.DataFrame({"before": open("agent_app/requirements.txt").read().splitlines(), "after": new_lines}))

In [ ]:
!syft dir:agent_app_fixed -q -o cyclonedx-json=sbom_fixed.cdx.json
!grype sbom:sbom_fixed.cdx.json -q -o json --file grype_fixed.json
after = json.load(open("grype_fixed.json"))["matches"]
sev = lambda ms: pd.Series([m["vulnerability"]["severity"] for m in ms]).value_counts().reindex(order).fillna(0).astype(int)
pd.DataFrame({"before": sev(g["matches"]), "after": sev(after)})

Some findings may remain. One fix can pull in another package with its own issue, or a CVE may have no fix yet (`fix_state = not-fixed`). That is normal. You then decide: accept the risk, add a guardrail, or replace the package.

## Step 8. Real world: scan the container your agent ships in
Agents usually run in Docker. The base image brings its own OS packages (openssl, glibc...). Syft pulls the image straight from the registry, no Docker needed.

In [ ]:
!syft python:3.10-slim -q -o cyclonedx-json=image_sbom.json
!grype sbom:image_sbom.json -q -o json --file image_grype.json
im = json.load(open("image_grype.json"))["matches"]
print("Packages in image:", len(json.load(open("image_sbom.json")).get("components", [])))
print("Findings by severity:"); print(sev(im).to_string())

## Production best practices for agentic AI
1. **Generate the SBOM at build time** and store it with the release. It is your record of exactly what went live.
2. **Re-scan stored SBOMs daily.** New CVEs are published every day; the code does not need to change for you to become vulnerable.
3. **Gate deployments** with `--fail-on high` in CI (GitHub Actions: `anchore/scan-action`).
4. **Scan the image, not only requirements.txt.** The OS layer often has more issues than Python packages.
5. **Pin versions and use lock files** so the SBOM matches what actually runs.
6. **Agents raise the stakes.** A vulnerable package in a normal app leaks data; in an agent with tools it can run code, send emails or move money. Treat agent dependencies as part of the attack surface (OWASP LLM Top 10: *Supply Chain*).
7. **Use `.grype.yaml` to record accepted risks** with a reason and an owner, instead of silently ignoring CVEs.
8. **Cover AI-specific parts too.** Syft lists code packages; model files and datasets need separate checks (hash pinning, trusted sources, safetensors instead of pickle).